# ----------------------------------------------------

# Getters and setters

## Manual (non-Pythonic) way

Let’s first build this in the manual (non-Pythonic) way, just like we used to do in traditional OOP.

1️⃣ Manual getter and setter example

Scenario:

We have a Customer with a full name, and we want to:
- get only the first name
- set/update only the first name

In [1]:
class Customer:
    def __init__(self, full_name, loyalty_points):
        self.full_name = full_name
        self.loyalty_points = loyalty_points

    # Getter method
    def get_first_name(self):
        parts = self.full_name.split(" ")
        return parts[0]

    # Setter method
    def set_first_name(self, new_first_name):
        parts = self.full_name.split(" ")
        updated_name = f"{new_first_name} {parts[1]}"
        self.full_name = updated_name


# Usage
c = Customer("Rahul Sharma", 120)

# Get first name
print(c.get_first_name())   # Rahul

# Update first name
c.set_first_name("Amit")

print(c.full_name)          # Amit Sharma

Rahul
Amit Sharma


2️⃣ Problems with this approach

❌ Not clean syntax

We would prefer:

In [ ]:
c.first_name        # instead of get_first_name()
c.first_name = "Amit"  # instead of set_first_name()

❌ Too many methods

For every attribute:
- `get_x()`
- `set_x()`

This becomes verbose in large systems.

❌ No control on direct access

Even now, someone can still do:

In [ ]:
c.full_name = "Invalid Data"

So control is not enforced properly.

## Pythonic `@property` version

### Pythonic `@property` version

Now let’s convert our manual getter/setter example into the Pythonic `@property` version and then discuss all the important rules and pitfalls.

Optimized version using `@property`

In [1]:
class Customer:
    def __init__(self, full_name, loyalty_points):
        self._full_name = full_name
        self.loyalty_points = loyalty_points

    @property
    def first_name(self):
        parts = self._full_name.split(" ")
        return parts[0]

    @first_name.setter
    def first_name(self, new_first_name):
        parts = self._full_name.split(" ")
        updated_name = f"{new_first_name} {parts[1]}"
        self._full_name = updated_name


# Usage
c = Customer("Rahul Sharma", 120)

print(c.first_name)      # getter
c.first_name = "Amit"    # setter
print(c._full_name)      # Amit Sharma

Rahul
Amit Sharma


What changed compared to manual version

| Manual way | Pythonic way |
| --- | --- |
| `get_first_name()` | `first_name` |
| `set_first_name(x)` | `first_name = x` |

👉 Cleaner, more natural, and feels like attribute access

### Very important rules to keep in mind

1️⃣ Names MUST match

In [ ]:
@property
def first_name(self): ...

In [ ]:
@first_name.setter
def first_name(self, value): ...

✔ Both must use same name: `first_name`

❌ This will NOT work:

In [ ]:
@first_name.setter
def set_first_name(self, value):  # ❌ wrong

---

2️⃣ Use a different internal variable

In [ ]:
self._full_name

👉 Why?

To avoid infinite recursion

❌ Wrong:

In [ ]:
self.full_name = full_name  # ❌ dangerous

This can trigger setter again and again.

✔ Correct:

In [ ]:
self._full_name = full_name

👉 Always use `_internal_variable`

---

3️⃣ Getter method name becomes the attribute

In [ ]:
c.first_name

👉 This calls:

In [ ]:
def first_name(self)

So: method name = attribute name

---

4️⃣ Setter must accept exactly one value

In [ ]:
def first_name(self, new_first_name):

✔ Only one extra parameter allowed

---

5️⃣ Order matters

Always define:

In [ ]:
@property

before:

In [ ]:
@first_name.setter

### Read-only property

If we only define:

In [ ]:
@property
def first_name(self):
    return ...

and DO NOT define setter:

In [ ]:
c.first_name = "Amit"

❌ Raises: AttributeError

👉 This makes attribute read-only

### When should we use `@property`?

Use it when:
- validation is needed
- computed value is needed
- controlled updates required
- backward compatibility (important in real systems)

👉 `@property` lets us change implementation WITHOUT changing usage

# ----------------------------------------------------

# Static methods | Class methods | Instance methods

1️⃣ Types of methods

In Python classes, we can have 3 types of methods:

| Type            | Works with        | First parameter | Use case                    |
| --------------- | ----------------- | --------------- | --------------------------- |
| Instance method | object (instance) | `self`          | works with object data      |
| Class method    | class             | `cls`           | works with class-level data |
| Static method   | neither           | none            | utility/helper logic        |


2️⃣ Types of attributes

Just like methods, we also have:

| Attribute type     | Where defined     | Accessed via    |
| ------------------ | ----------------- | --------------- |
| Instance attribute | inside `__init__` | object          |
| Class attribute    | directly in class | class or object |

👉 There is no “static attribute” keyword in Python

But class attributes behave like static variables.

3️⃣ One complete example

In [2]:
class Order:
    # CLASS ATTRIBUTE (shared by all objects)
    restaurant_name = "FoodHub"

    def __init__(self, item, price):
        # INSTANCE ATTRIBUTES (unique per object)
        self.item = item
        self.price = price

    # 1️⃣ INSTANCE METHOD
    # Always has self, Works with instance attributes, Called using object
    # 👉 Think: "This method belongs to a specific object"
    def get_order_details(self):
        # self refers to the current object
        return f"Item: {self.item}, Price: {self.price}"
        # this can access and modify Instance attributes, and Class attributes (but careful ⚠️)

    # 2️⃣ CLASS METHOD
    # Uses @classmethod decorator, First parameter is cls (class reference), Works with class attributes, 
    # Can modify shared data
    # 👉 Think: "This method belongs to the class itself"
    @classmethod
    def change_restaurant_name(cls, new_name):
        # cls refers to the class itself
        cls.restaurant_name = new_name
        # this can only access and modify Class attributes

    # 3️⃣ STATIC METHOD
    # Uses @staticmethod, No self, no cls, Just a normal function inside class namespace
    # 👉 Think: "This is just a helper function grouped inside the class"
    @staticmethod
    def calculate_discount(price, discount_percent):
        # No self, no cls → pure logic
        return price - (price * discount_percent / 100)
        # this cannot access Instance attributes, or Class attributes


# ---------------- USAGE ----------------

# Create objects
o1 = Order("Pizza", 500)
o2 = Order("Burger", 200)

# INSTANCE METHOD
print(o1.get_order_details())  # Item: Pizza, Price: 500
# Uses object data (self), Called using object

# CLASS ATTRIBUTE access
print(Order.restaurant_name)  # FoodHub
print(o1.restaurant_name)     # FoodHub
# Both work (shared data)

# CLASS METHOD
Order.change_restaurant_name("SuperFoodHub")

print(o1.restaurant_name)  # SuperFoodHub   # changed for all 
print(o2.restaurant_name)  # SuperFoodHub

# STATIC METHOD
discounted_price = Order.calculate_discount(500, 10)
print(discounted_price)    # 450.0

Item: Pizza, Price: 500
FoodHub
FoodHub
SuperFoodHub
SuperFoodHub
450.0


4️⃣ Key differences (very important)

| Feature               | Instance       | Class | Static |
| --------------------- | -------------- | ----- | ------ |
| Access to object data | ✅              | ❌     | ❌      |
| Access to class data  | ✅ (indirectly) | ✅     | ❌      |
| Needs decorator       | ❌              | ✅     | ✅      |
| First parameter       | self           | cls   | none   |


5️⃣ Very important subtle behavior ⚠️

Case: modifying class attribute from instance


In [ ]:
self.restaurant = "X"

<pre>
👉 This does NOT modify class attribute
👉 It creates a new instance attribute
</pre>
So:

In [ ]:
print(Order.restaurant)  # unchanged
print(self.restaurant)   # new value

<pre>
Instance method → "I know MY data + shared data"
Class method → "I know shared data only"
Static method → "I know nothing unless given"
</pre>

<br>
6️⃣ Common pitfalls ⚠️

❌ Accessing instance attribute from class

In [ ]:
print(Order.item)  # ❌ ERROR

Because: `item` belongs to object, not class

---

❌ Modifying class attribute incorrectly

In [ ]:
o1.restaurant_name = "X"

👉 Creates new attribute only for `o1`, does NOT change class

---

❌ Forgetting decorators

In [ ]:
def my_method():

👉 Without `@classmethod` or `@staticmethod`, Python treats it as instance method

---

7️⃣ When to use what

Use instance method when:
- logic depends on object data

Use class method when:
- modifying class-level data
- factory methods (advanced use)

Use static method when:
- utility logic related to class
- but no need for object/class access

8️⃣ Real-world intuition

<pre>
Instance method → "this order"
Class method → "all orders"
Static method → "helper related to orders"
</pre>

# ----------------------------------------------------

# Magic/Dunder Methods

## 1️⃣ What are magic (dunder) methods?

👉 “Dunder” = Double UNDERscore

These are methods like:
<pre>
__init__
__str__
__len__
__add__
</pre>

Core idea: Magic methods let us define how our objects behave with built-in Python operations

Example intuition

When we write:

In [ ]:
print(obj)

Python internally does:

In [ ]:
obj.__str__()

Why are they useful?

They allow us to:
- customize object creation
- define how objects behave with operators (`+`, `==`, etc.)
- control printing/logging
- make objects behave like lists/dictionaries
- control attribute access

## 2️⃣ Most important categories

🔹 1. Object creation & initialization | `__init__`

In [3]:
class Order:
    def __init__(self, item, price): # 👉 Runs automatically when object is created
        self.item = item
        self.price = price

order1 = Order("pizza", 1500) # Here, we did not call the `__init__` method explicitly
print(order1.item, order1.price)

pizza 1500


🔹 2. String representation | `__str__`

In [4]:
class Order:
    def __init__(self, item, price):
        self.item = item
        self.price = price

    def __str__(self): # 👉 Instead of ugly object memory address, we get readable output
        return f"Order: {self.item} costs ₹{self.price}"
    
order1 = Order("Pizza", 500)
print(order1)

Order: Pizza costs ₹500


In [5]:
class Order:
    def __init__(self, item, price):
        self.item = item
        self.price = price

    # def __str__(self):
    #     return f"Order: {self.item} costs ₹{self.price}"
    
order1 = Order("Pizza", 500) 
print(order1) # lets check the output without the `__str__` dunder method

🔹 3. Length behavior | `__len__`

In [6]:
class Cart:
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)
    
cart = Cart(["Pizza", "Burger"])
print(len(cart))   # calls __len__

2


🔹 4. Operator overloading | `__add__`

In [7]:
class Order:
    def __init__(self, price):
        self.price = price

    def __add__(self, other):
        return Order(self.price + other.price)
    
o1 = Order(300)
o2 = Order(200)

o3 = o1 + o2   # calls __add__
print(o3.price)

500


Other operators

| Operator | Method    |
| -------- | --------- |
| `+`      | `__add__` |
| `-`      | `__sub__` |
| `*`      | `__mul__` |
| `==`     | `__eq__`  |
| `<`      | `__lt__`  |


🔹 5. Comparison methods | `__eq__`

In [8]:
class Order:
    def __init__(self, price):
        self.price = price

    def __eq__(self, other):
        return self.price == other.price
    
o1 = Order(300)
o2 = Order(300)

print(o1 == o2)   # True (calls __eq__)

True


🔹 6. Attribute control | `__getattr__` and `__setattr__`

In [10]:
class Customer:
    def __init__(self, name):
        self.name = name

    def __getattr__(self, attr):
        return f"{attr} not found"

c = Customer("Rahul")
print(c.name)
print(c.age)   # handled by __getattr__
# ⚠️ Note: __getattr__ is called ONLY when attribute is NOT found

Rahul
age not found


⚠️ Important: `__setattr__` is powerful but dangerous (can cause recursion if misused)

<details>
<summary>Why `__getattr__` is called ONLY when attribute is NOT found?</summary>
1️⃣ What actually happens when we do c.name?

When we access: `c.name`

Python follows a lookup order:
<pre>
1️⃣ Check instance dictionary (c.__dict__)
2️⃣ Check class attributes
3️⃣ Call __getattr__ (ONLY if not found above)
</pre>

2️⃣ In our example

```python
class Customer:
    def __init__(self, name):
        self.name = name
```

When object is created: `c = Customer("Rahul")`

Internally:

```python
c.__dict__ = {
    "name": "Rahul"
}
```

So when we do: `print(c.name)`

👉 Python finds "`name`" in: `c.__dict__`

- ✔ Found immediately
- ❌ So `__getattr__` is NOT called

</details>

🔹 7. Callable objects | `__call__`

In [11]:
class Discount:
    def __init__(self, percent):
        self.percent = percent

    def __call__(self, amount):
        return amount - (amount * self.percent / 100)
    
d = Discount(10)
print(d(500))   # object behaves like function

450.0


🔹 8. Container behavior | `__getitem__` 

In [12]:
class Menu:
    def __init__(self, items):
        self.items = items

    def __getitem__(self, index):
        return self.items[index]
    
menu = Menu(["Pizza", "Burger"])
print(menu[0])   # like list access

Pizza


Related methods

| Behavior          | Method         |
| ----------------- | -------------- |
| indexing          | `__getitem__`  |
| setting value     | `__setitem__`  |
| deleting          | `__delitem__`  |
| membership (`in`) | `__contains__` |


🔹 9. Context managers | `__enter__` and `__exit__`

In [13]:
class FileHandler:
    def __enter__(self):
        print("Opening resource")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        print("Closing resource")

with FileHandler() as f:
    print("Using resource")

Opening resource
Using resource
Closing resource


---

Real-world intuition

<pre>
__init__  → "how object is created"
__str__   → "how object looks"
__add__   → "how object behaves with +"
__len__   → "how object behaves with len()"
__call__  → "can object act like function?"
</pre>

🔹 10. `__repr__` | representation

👉 repr = representation

`__repr__` → "representation of the object"

In [16]:
class Order:
    def __init__(self, item, price):
        self.item = item
        self.price = price

    def __repr__(self):
        return f"Order(item='{self.item}', price={self.price})"

    def __str__(self):
        return f"{self.item} costs ₹{self.price}"
    
o = Order("Pizza", 500)

print(o)     # uses __str__
o            # uses __repr__ (in console)

Pizza costs ₹500


Order(item='Pizza', price=500)

What if `__str__` is missing?

Then Python falls back to: `__repr__`

Difference summary

| Method     | Audience   | Purpose                |
| ---------- | ---------- | ---------------------- |
| `__repr__` | Developers | Debugging, unambiguous |
| `__str__`  | Users      | Readable display       |


---

## 3️⃣ Important rules and pitfalls to remember

✔ These methods are automatically called

We never call: `obj.__str__()`

Instead: `print(obj)`

---

✔ Names must match exactly
```python
__init__   # correct
__Init__   # ❌ wrong
```

---
✔ Not all need to be implemented

We only define what we need.

---
Common pitfalls ⚠️

❌ Infinite recursion
```python
def __str__(self):
    return str(self)   # ❌ calls itself
```

---
❌ Forgetting return type
```python
def __len__(self):
    return "5"   # ❌ must return int
```
---
❌ Not handling type in operator overloading
```python
def __add__(self, other):
    return self.price + other.price   # may crash if other is not same type
```
Better:
```python
if not isinstance(other, Order):
    return NotImplemented
```
---

# ----------------------------------------------------

# Exception Handling and throw Custom Errors

## Exception Handling

1️⃣ The problem: What happens without exception handling?

Scenario (Restaurant system 🍽️)

We are building a function to calculate price per person.

In [17]:
def calculate_price_per_person(total_amount, number_of_people):
    return total_amount / number_of_people


# Usage
amount = 1000
people = 0

result = calculate_price_per_person(amount, people)
print(result)

ZeroDivisionError: division by zero

❌ What goes wrong? 

ZeroDivisionError: division by zero

👉 Problem:
- Program crashes ❌
- No graceful handling ❌
- User sees ugly error ❌

2️⃣ Why this is a problem in real systems

Imagine:
- Payment system crashes ❌
- API fails without response ❌
- UI breaks ❌

👉 This is unacceptable in production systems.

3️⃣ Solution: Exception Handling

Python gives us a way to: Handle errors gracefully instead of crashing

We need exception handling to prevent program crashes and handle unexpected situations gracefully.

4️⃣ Basic exception handling (`try-except`)

In [18]:
def calculate_price_per_person(total_amount, number_of_people):
    try:
        # Risky operation
        return total_amount / number_of_people

    except ZeroDivisionError:
        # Handle specific error
        print("Number of people cannot be zero.")
        return None


# Usage
result = calculate_price_per_person(1000, 0)
print(result)

Number of people cannot be zero.
None


✅ Now what happens?
- Program does NOT crash ✔️
- Error is handled ✔️
- User gets meaningful message ✔️

5️⃣ Multiple exception handling



In [ ]:
def calculate_price_per_person(total_amount, number_of_people):
    try:
        return total_amount / number_of_people

    except ZeroDivisionError:
        print("Cannot divide by zero.")

    except TypeError:
        print("Invalid input type. Please provide numbers.")

6️⃣ Catching all exceptions (careful ⚠️)

In [ ]:
def safe_division(a, b):
    try:
        return a / b
    except Exception as error:
        print(f"Something went wrong: {error}")

👉 Use this carefully — don’t hide real issues in production.

7️⃣ `else` and `finally`

In [19]:
def process_payment(amount):
    try:
        print("Processing payment...")
        result = amount / 2

    except Exception:
        print("Payment failed.")

    else:
        print("Payment successful.")

    finally:
        print("Transaction attempt completed.")

    return result


process_payment(100)

Processing payment...
Payment successful.
Transaction attempt completed.


50.0

Meaning:

| Block     | When it runs    |
| --------- | --------------- |
| `try`     | main logic      |
| `except`  | if error occurs |
| `else`    | if NO error     |
| `finally` | always runs     |


## Custom Errors

1️⃣ Raising exceptions manually (raise)

Sometimes we want to trigger errors ourselves.

Example: Validation

In [20]:
def create_order(amount):
    if amount <= 0:
        raise ValueError("Order amount must be greater than zero.")

    return f"Order created with amount ₹{amount}"


# Usage
print(create_order(-100))

ValueError: Order amount must be greater than zero.

2️⃣ Why custom errors are needed

Built-in errors are sometimes too generic.

Example:

ValueError → but what value? why?

👉 We want domain-specific clarity.

3️⃣ Custom Exception

In [21]:
# Custom exception class
class InvalidOrderAmountError(Exception):
    """Raised when order amount is invalid"""
    pass


def create_order(amount):
    if amount <= 0:
        raise InvalidOrderAmountError(
            f"Invalid order amount: {amount}. Must be greater than zero."
        )

    return f"Order created with amount ₹{amount}"


# Usage
try:
    create_order(-50)
except InvalidOrderAmountError as error:
    print(f"Custom Error Caught: {error}")

Custom Error Caught: Invalid order amount: -50. Must be greater than zero.


✅ Benefits
- Clear domain-specific errors ✔️
- Easier debugging ✔️
- Better API design ✔️

4️⃣ Real-world structured example

In [22]:
class PaymentFailedError(Exception):
    """Custom exception for payment failure"""
    pass


def process_payment(amount):
    try:
        if amount > 5000:
            raise PaymentFailedError("Payment exceeds allowed limit.")

        print("Payment processed successfully.")

    except PaymentFailedError as error:
        print(f"Payment Error: {error}")

    finally:
        print("Payment attempt logged.")


# Usage
process_payment(6000)

Payment Error: Payment exceeds allowed limit.
Payment attempt logged.


# ----------------------------------------------------

# ----------------------------------------------------

# ----------------------------------------------------

# ----------------------------------------------------